# SIBI Dynamic Point-History Classifier (Phase 2F)

Trains a small MLP that maps a 24-frame wrist trajectory (24 × 2 = 48 floats) to one of 5 SIBI word classes (jeruk, kucing, besar, kecil, sangat).

Input CSV: `point_history_csv/dynamic.csv` (collected via `/dev/gesture-recorder`).
Output: `point_history_classifier.hdf5`, `point_history_classifier.tflite`.

In [ ]:
import csv
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
NUM_CLASSES = 5
FEATURE_LENGTH = 48  # 24 frames * 2 coords
CSV_PATH = 'point_history_csv/dynamic.csv'
LABEL_CSV_PATH = 'point_history_csv/dynamic_label.csv'
TFLITE_PATH = 'point_history_classifier.tflite'
HDF5_PATH = 'point_history_classifier.hdf5'

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Total samples: {len(df)}')
print(f'Samples per class:\n{df["label"].value_counts()}')

labels = pd.read_csv(LABEL_CSV_PATH, header=None)[0].tolist()
label_to_idx = {l: i for i, l in enumerate(labels)}
assert len(labels) == NUM_CLASSES

X = df.iloc[:, 1:].astype(np.float32).values
y_strings = df['label'].astype(str).values
y = np.array([label_to_idx[l] for l in y_strings], dtype=np.int32)
assert X.shape[1] == FEATURE_LENGTH
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Lightweight MLP — Kazuhito uses LSTM for dynamic, but for fixed-length
# 24-frame windows a dense MLP is simpler and fast enough.
model = tf.keras.models.Sequential([
    tf.keras.layers.Input((FEATURE_LENGTH,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    HDF5_PATH, verbose=1, save_weights_only=False, save_best_only=True,
)
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback],
    verbose=2,
)

In [ ]:
model = tf.keras.models.load_model(HDF5_PATH)
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred, target_names=labels, digits=3))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('SIBI Dynamic Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized = converter.convert()
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_quantized)
print(f'Wrote {TFLITE_PATH} ({len(tflite_quantized)} bytes)')